<a href="https://colab.research.google.com/github/Davidson-Open-University/phy107-labs/blob/main/notebooks/PHY107-Lab6-Pressure-Flow-and-Viscosity.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# PHY 107 Lab 6: Pressure, Flow and Viscosity

**Unit 6: Viscosity and fluids** · General Practical Physics I · Davidson Open University Global

**In this lab you will:**
- calculate pressure at a depth in a liquid;
- use the continuity equation and flow rate;
- find viscosity by Stokes' falling-ball method;
- classify flow as laminar or turbulent with the Reynolds number.

**Time:** about 60 minutes in four parts:
- **Part A – Explore** (about 10 minutes): run the online lab and the examples, and read the explanations.
- **Part B – Practise** (about 20 minutes): fill in the blanks. A self-check tells you if you are right.
- **Part C – Build a tool** (about 20 minutes): complete a short analysis function and test it on data.
- **Part D – Your own data and reflection** (about 10 minutes): analyse readings you took in the online lab (or at home), and write short answers.

**How to use this notebook**
1. Run each code cell in order: click it and press **Shift + Enter** (or the ▶ button).
2. Where you see `...`, replace it with your own code, run the cell, then run the **check** cell under it.
   - ✅ means correct. ❌ gives you a hint: try again. ⏳ means you have not filled in the blank yet.
   - The checks never show the answer, so you learn by working it out. You can try as many times as you like.
3. Stuck? Read the hint, look back at the study session, or ask in the discussion forum. Do not skip ahead: each part builds on the last.
4. Save your work: in Colab choose **File → Save a copy in GitHub** and save it to your own `phy107-labs` repository. Offline in Jupyter, press **Ctrl + S** and later upload the file with **Add file → Upload files** on GitHub.

**Honest data:** keep your own readings separate from the example data, and label every simulated or supplied reading as such. Never invent readings.

*This lab is practice to help you learn and prepare for the unit assignment (lab report). It is not graded unless your tutor tells you otherwise.*

In [ ]:
#@title ▶ Run this cell first: it sets up the self-checks (you do not need to read it)
# You do not need to read or change this code.
import hashlib as _h
class _Blank(Exception): pass
def _norm(x, unordered=False, dp=6):
    try:
        import numpy as _np
        if isinstance(x, _np.ndarray): x = x.tolist()
        elif isinstance(x, _np.bool_): x = bool(x)
        elif isinstance(x, _np.integer): x = int(x)
        elif isinstance(x, _np.floating): x = float(x)
    except ImportError:
        pass
    if x is Ellipsis or x is None: raise _Blank()
    if isinstance(x, bool): return 'b:' + str(x)
    if isinstance(x, (int, float)): return 'n:' + repr(round(float(x), dp) + 0.0)
    if isinstance(x, str): return 't:' + ' '.join(x.lower().split())
    if isinstance(x, (set, frozenset)): return 's{' + ','.join(sorted(_norm(i, False, dp) for i in x)) + '}'
    if isinstance(x, (list, tuple)):
        items = [_norm(i, False, dp) for i in x]
        if unordered: items.sort()
        return 'l[' + ','.join(items) + ']'
    if isinstance(x, dict): return 'd{' + ','.join(sorted(_norm(k, False, dp) + '=' + _norm(v, False, dp) for k, v in x.items())) + '}'
    try:
        import sympy as _sp
        if isinstance(x, _sp.Basic):
            x = _sp.sympify(x)
            if x.is_number: return 'n:' + repr(round(float(x), dp) + 0.0)
            syms = sorted(x.free_symbols, key=str)
            vals = []
            for t in (0.3, 0.7, 1.3):
                v = complex(x.subs({s: t + 0.11 * i for i, s in enumerate(syms)}).evalf())
                vals.append(round(v.real, 6) + 0.0)
            return 'e:' + repr(vals)
    except ImportError:
        pass
    return 'r:' + repr(x)
def _hash(q, s): return _h.sha256(('phy107|' + q + '|' + s).encode()).hexdigest()[:16]
def check(q, value):
    """Check your answer to question q. Prints ✅, ❌ with a hint, or ⏳ if the blank is not filled yet."""
    spec = _CHECKS[q]
    try:
        if callable(value) and 'args' in spec:
            outs = []
            for a in spec['args']:
                try:
                    outs.append(value(*a))
                except _Blank:
                    raise
                except Exception as e:
                    print(f'❌ Question {q}: your function stopped with an error: {type(e).__name__}: {e}')
                    return
            if all(o is None or o is Ellipsis for o in outs): raise _Blank()
            got = _norm(outs, False, spec.get('dp', 6))
        elif spec.get('logic'):
            if value is None or value is Ellipsis: raise _Blank()
            import sympy as _sp
            from itertools import product as _p
            syms = sorted(_sp.sympify(value).free_symbols, key=str)
            got = 'L' + ''.join('1' if bool(_sp.sympify(value).subs(dict(zip(syms, v)))) else '0' for v in _p([True, False], repeat=len(syms))) + ':' + ','.join(map(str, syms))
        else:
            got = _norm(value, spec.get('unordered', False), spec.get('dp', 6))
    except _Blank:
        print(f'⏳ Question {q}: fill in the blank (replace ...) and run the cell above, then run this cell again.')
        return
    if _RECORD is not None:
        _RECORD[q] = _hash(q, got); print(f'✅ Question {q}: recorded.'); return
    if _hash(q, got) == spec['h']:
        print(f'✅ Question {q}: correct! ' + spec.get('ok', 'Well done.'))
    else:
        print(f'❌ Question {q}: not yet. Hint: ' + spec['hint'])
_RECORD = None

_CHECKS = {
    '6.1': {'h': '4cc29a9ad429b599', 'hint': 'p = ρ * g * h.', 'dp': 0},
    '6.2': {'h': '02d9bc84b262795b', 'hint': 'A₁v₁ = A₂v₂, so v₂ = A₁v₁ / A₂ (the units of area cancel).', 'ok': 'Narrow the pipe, and the water speeds up, like a thumb over a hose.', 'dp': 2},
    '6.3': {'h': '0db8e8737d646f4f', 'hint': 'Q = A * v.', 'dp': 6},
    '6.4': {'h': 'ecd3fdf06dcbf683', 'hint': 'np.polyfit(time, dist, 1)[0]', 'dp': 4},
    '6.5': {'h': 'e10d385f82a86fbe', 'hint': 'η = 2 * r**2 * (rho_s - rho_f) * 9.81 / (9 * v).', 'ok': 'Glycerine is about a thousand times more viscous than water.', 'dp': 2},
    '6.6': {'h': '70009e1ba67f79cd', 'hint': 'Re = ρ v d / η; laminar below 2000, turbulent above 4000.', 'dp': 0},
    '6.7': {'h': 'f49bbf81e6f090e6', 'hint': 'Watch the brackets: (rho_s - rho_f) and (9 * v).', 'ok': 'Your viscosity tool works.', 'dp': 3, 'args': [(1.0e-3, 7800, 1260, 0.012), (0.5e-3, 7800, 1260, 0.003), (1.5e-3, 2500, 900, 0.02)]},
}
print('Self-checks ready. Now carry on with Part A.')

## Part A – Explore (about 10 minutes)
### 🔬 Online lab for this unit (free open educational resources)
- **[PhET Under Pressure](https://phet.colorado.edu/en/simulations/under-pressure)** – measure pressure at different depths and in different liquids.
- **[PhET Fluid Pressure and Flow](https://phet.colorado.edu/en/simulations/fluid-pressure-and-flow)** – change the pipe width and watch the flow speed (continuity).
- **[Viscosity by Stokes' method (Virtual Labs, NITK Surathkal)](https://ph1-nitk.vlabs.ac.in/exp/viscosity-stokes/)** – drop balls through a liquid and time their terminal velocity. Free.

Open the online lab in a new tab, try it for a few minutes, then come back here. **Label every reading you take there as "simulated".**

**Key results.** Pressure at depth: p = ρgh. Continuity: A₁v₁ = A₂v₂. Flow rate: Q = Av. Stokes' method: a small sphere reaches a terminal velocity v, and the viscosity is η = 2r²(ρₛ − ρ_f)g / (9v). Reynolds number: Re = ρvd/η (below about 2000 laminar, above about 4000 turbulent).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
rho, g = 1000, 9.81
for h in [0.5, 1, 2]:
    print(f'depth {h} m  ->  extra pressure {rho * g * h:.0f} Pa')

## Part B – Practise (about 20 minutes)
**B1.** Find the extra pressure (Pa) 2.0 m below the surface of water (ρ = 1000 kg/m³).

In [ ]:
answer_6_1 = ...

In [ ]:
check('6.1', answer_6_1)

In [ ]:
answer_6_2 = ...   # water at 1.5 m/s flows from a 4.0 cm² pipe into a 1.0 cm² pipe: new speed in m/s

In [ ]:
check('6.2', answer_6_2)

In [ ]:
answer_6_3 = ...   # flow rate Q in m³/s for A = 4.0e-4 m² and v = 1.5 m/s

In [ ]:
check('6.3', answer_6_3)

**B4. Terminal velocity from data.** A steel ball falls through glycerine (illustrative data). Once it reaches terminal velocity, distance increases steadily with time. Find v (m/s) as the gradient of distance against time, 4 decimal places.

In [ ]:
time = np.array([0.0, 2.0, 4.0, 6.0, 8.0])           # s   (illustrative, after reaching terminal velocity)
dist = np.array([0.000, 0.024, 0.048, 0.073, 0.096]) # m
plt.plot(time, dist, 'o'); plt.xlabel('time (s)'); plt.ylabel('distance (m)'); plt.show()

In [ ]:
answer_6_4 = ...

In [ ]:
check('6.4', answer_6_4)

**B5. Stokes' viscosity.** Use r = 1.0 mm (1.0e-3 m), ρₛ = 7800 kg/m³ (steel), ρ_f = 1260 kg/m³ (glycerine) and your v from B4. Find η in Pa s (2 decimal places).

In [ ]:
answer_6_5 = ...

In [ ]:
check('6.5', answer_6_5)

**B6. Reynolds number.** Water (ρ = 1000 kg/m³, η = 1.0e-3 Pa s) flows at 0.50 m/s in a pipe of diameter 0.010 m. Give `(Re, regime)` where regime is `"laminar"`, `"transitional"` or `"turbulent"`.

In [ ]:
Re = ...
regime = ...
answer_6_6 = (Re, regime)

In [ ]:
check('6.6', answer_6_6 if ... not in answer_6_6 else ...)

## Part C – Build a tool (about 20 minutes)
Complete `stokes_viscosity(r, rho_s, rho_f, v)` so that it returns η = 2r²(ρₛ − ρ_f)g / (9v), with g = 9.81, rounded to 3 decimal places.

In [ ]:
def stokes_viscosity(r, rho_s, rho_f, v):
    ...      # YOUR CODE

In [ ]:
check('6.7', stokes_viscosity)

## Part D – Your own data and reflection (about 10 minutes)
**D1. Prepares you for the Unit 6 assignment.** In the NITK virtual lab, record distance–time data for two ball sizes and find η with your tool. Or, in PhET Fluid Pressure and Flow, record pipe areas and speeds and test A₁v₁ = A₂v₂. Label the data "simulated".

In [ ]:
my_time = [...]
my_dist = [...]

**D2. Reflect.**
1. Why must the ball reach terminal velocity before you start timing?
2. How would warming the liquid change the falling speed? Why?
3. Give one place in daily life where flow becomes turbulent.

*Type your answer here (double-click this cell to edit it).*

## Finished? Save your work
In Colab: **File → Save a copy in GitHub** → your `phy107-labs` repository → commit message *"Lab 6 completed"*.
Offline: save with **Ctrl + S**, then upload the file to your repository with **Add file → Upload files** when you are online.

Then go back to Moodle and mark the lab as done.